In [ ]:
# Task 5: Text Preprocessing and Embedding Generation using Hugging Face Transformers

## Objective
To preprocess text descriptions into tokenized and encoded representations using the Hugging Face Transformers library. These embeddings can later be used as inputs for text-to-image generation models.

In [6]:
!pip install transformers torch

In [7]:
import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd

In [8]:
captions = [
    "A dog running in the park",
    "A red car parked on the road",
    "A child playing football",
    "A beautiful sunset over the mountains",
    "Two cats sleeping on a sofa"
]

df = pd.DataFrame(captions, columns=["Caption"])

df

,Caption
0,A dog running in the park
1,A red car parked on the road
2,A child playing football
3,A beautiful sunset over the mountains
4,Two cats sleeping on a sofa


In [9]:
def clean_text(text):
    text = text.lower()
    return text

df["Cleaned_Caption"] = df["Caption"].apply(clean_text)

df

,Caption,Cleaned_Caption
0,A dog running in the park,a dog running in the park
1,A red car parked on the road,a red car parked on the road
2,A child playing football,a child playing football
3,A beautiful sunset over the mountains,a beautiful sunset over the mountains
4,Two cats sleeping on a sofa,two cats sleeping on a sofa


In [10]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer Loaded Successfully")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer Loaded Successfully


In [11]:
tokens = tokenizer(
    df["Cleaned_Caption"].tolist(),
    padding=True,
    truncation=True,
    return_tensors="pt"
)

print(tokens.keys())

KeysView({'input_ids': tensor([[  101,  1037,  3899,  2770,  1999,  1996,  2380,   102,     0],
        [  101,  1037,  2417,  2482,  9083,  2006,  1996,  2346,   102],
        [  101,  1037,  2775,  2652,  2374,   102,     0,     0,     0],
        [  101,  1037,  3376, 10434,  2058,  1996,  4020,   102,     0],
        [  101,  2048,  8870,  5777,  2006,  1037, 10682,   102,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0]])})


In [12]:
model = AutoModel.from_pretrained(model_name)

with torch.no_grad():
    outputs = model(**tokens)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
embeddings = outputs.last_hidden_state

print("Embedding Shape:")
print(embeddings.shape)

Embedding Shape:
torch.Size([5, 9, 768])


In [14]:
sentence_embeddings = embeddings.mean(dim=1)

print(sentence_embeddings.shape)

torch.Size([5, 768])


In [15]:
torch.save(sentence_embeddings, "text_embeddings.pt")

print("Embeddings Saved Successfully")

Embeddings Saved Successfully


In [18]:
print("""
Conclusion

The text descriptions were successfully preprocessed using Hugging Face Transformers.

The DistilBERT tokenizer converted the text into tokenized inputs, and the DistilBERT model generated 768-dimensional embeddings for each caption.

These embeddings can be used as inputs for future text-to-image generation models such as GANs or diffusion models to ensure accurate representation of textual information.
""")


Conclusion

The text descriptions were successfully preprocessed using Hugging Face Transformers.

The DistilBERT tokenizer converted the text into tokenized inputs, and the DistilBERT model generated 768-dimensional embeddings for each caption.

These embeddings can be used as inputs for future text-to-image generation models such as GANs or diffusion models to ensure accurate representation of textual information.



In [19]:
print("First Caption:")
print(df["Caption"][0])

print("\nEmbedding Vector Shape:")
print(sentence_embeddings[0].shape)

print("\nFirst 10 Values:")
print(sentence_embeddings[0][:10])

First Caption:
A dog running in the park

Embedding Vector Shape:
torch.Size([768])

First 10 Values:
tensor([-0.0151, -0.1636, -0.0975,  0.1109,  0.4191, -0.3766, -0.0026,  0.3859,
        -0.2977,  0.1884])


In [20]:
import pandas as pd

embeddings_df = pd.DataFrame(sentence_embeddings.numpy())

embeddings_df.to_csv("text_embeddings.csv", index=False)

print("Embeddings saved as text_embeddings.csv")

Embeddings saved as text_embeddings.csv


In [22]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [23]:
embeddings_df.to_csv(
    "/content/drive/MyDrive/text_embeddings.csv",
    index=False
)

torch.save(
    sentence_embeddings,
    "/content/drive/MyDrive/text_embeddings.pt"
)

In [26]:
user_text = input("Enter a text description: ")

tokens = tokenizer(
    [user_text],
    padding=True,
    truncation=True,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(**tokens)

embedding = outputs.last_hidden_state.mean(dim=1)

print("\nUser Input:")
print(user_text)

print("\nEmbedding Shape:")
print(embedding.shape)

Enter a text description: a cat on the chair

User Input:
a cat on the chair

Embedding Shape:
torch.Size([1, 768])
